Proposition 1 — Ranking Customer Orders by Value

Outer Query:

- Retrieves each customer's order along with the total order value.
It ranks orders within each customer by total discounted revenue.

Logic:

- Uses window functions like RANK() and DENSE_RANK() partitioned by CustomerId and ordered by total value descending.

Result:

- Shows order ranking and removes duplicates by dense rank for tied totals.


In [ ]:
%%sql
SELECT
    c.CustomerCompanyName,
    o.OrderId,
    SUM(od.UnitPrice * od.Quantity * (1 - od.DiscountPercentage)) AS TotalValue,
    RANK()       OVER(PARTITION BY c.CustomerId ORDER BY SUM(od.UnitPrice * od.Quantity * (1 - od.DiscountPercentage)) DESC) AS OrderRank,
    DENSE_RANK() OVER(PARTITION BY c.CustomerId ORDER BY SUM(od.UnitPrice * od.Quantity * (1 - od.DiscountPercentage)) DESC) AS DenseRank
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
GROUP BY c.CustomerCompanyName, c.CustomerId, o.OrderId;


Proposition 2 — Identify First Order per Employee

Outer Query:

- Lists every order made by employees and assigns a sequential number to each based on the order date.

Logic:

- Applies the ROW_NUMBER() window function partitioned by employee to label earliest orders.

Result:

- Highlights the first order for every employee.

In [ ]:
%%sql
SELECT
    e.EmployeeId,
    e.EmployeeLastName,
    o.OrderId,
    o.OrderDate,
    ROW_NUMBER() OVER(PARTITION BY e.EmployeeId ORDER BY o.OrderDate ASC) AS RowNum
FROM HumanResources.Employee AS e
JOIN Sales.[Order] AS o ON e.EmployeeId = o.EmployeeId;


Proposition 3 — Running Total of Order Value by Month

Outer Query:

- Aggregates total order value by month and year.

Logic:

- Uses a windowed SUM() function to calculate the running total over time.

Result:

- Displays each month’s total and cumulative revenue up to that point.

In [ ]:
%%sql
SELECT
    YEAR(o.OrderDate) AS OrderYear,
    MONTH(o.OrderDate) AS OrderMonth,
    SUM(od.UnitPrice * od.Quantity) AS MonthlySales,
    SUM(SUM(od.UnitPrice * od.Quantity)) OVER(ORDER BY YEAR(o.OrderDate), MONTH(o.OrderDate)) AS RunningTotal
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
GROUP BY YEAR(o.OrderDate), MONTH(o.OrderDate)
ORDER BY OrderYear, OrderMonth;


Proposition 4 — Difference Between Current and Previous Order Value

Outer Query:

- Retrieves each customer’s order values.

Logic:

- Uses LAG() and LEAD() to compare the current order with the previous and next ones.

Result:

- Helps detect spending trends or outliers in each customer’s order sequence.

In [ ]:
%%sql
SELECT
    c.CustomerCompanyName,
    o.OrderId,
    SUM(od.UnitPrice * od.Quantity) AS OrderValue,
    SUM(od.UnitPrice * od.Quantity) -
        LAG(SUM(od.UnitPrice * od.Quantity)) OVER(PARTITION BY c.CustomerId ORDER BY o.OrderDate) AS DiffPrev,
    LEAD(SUM(od.UnitPrice * od.Quantity)) OVER(PARTITION BY c.CustomerId ORDER BY o.OrderDate) -
        SUM(od.UnitPrice * od.Quantity) AS DiffNext
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
GROUP BY c.CustomerCompanyName, c.CustomerId, o.OrderId, o.OrderDate;


Proposition 5 — Pivot Order Counts by Year

Outer Query:

- Extracts employee names and order years.

Logic:

- Transforms rows into columns using the PIVOT operator, one column per year.

Result:

- Produces a matrix showing how many orders each employee handled per year.

In [ ]:
%%sql
SELECT *
FROM (
    SELECT
        YEAR(o.OrderDate) AS OrderYear,
        e.EmployeeLastName
    FROM Sales.[Order] AS o
    JOIN HumanResources.Employee AS e ON o.EmployeeId = e.EmployeeId
) AS src
PIVOT (
    COUNT(OrderYear)
    FOR OrderYear IN ([2020], [2021], [2022], [2023])
) AS p;


Proposition 6 — Unpivot Employee Order Counts

Outer Query:
- Takes pivoted data from the previous query and transforms columns back into rows.

Logic:

- Uses UNPIVOT to normalize the table structure.

Result:

- Each row now represents one employee-year combination.

In [ ]:
%%sql
SELECT EmployeeLastName, OrderYear, NumOrders
FROM (
    SELECT
        EmployeeLastName, [2020], [2021], [2022], [2023]
    FROM (
        SELECT
            e.EmployeeLastName,
            YEAR(o.OrderDate) AS OrderYear
        FROM Sales.[Order] AS o
        JOIN HumanResources.Employee AS e ON o.EmployeeId = e.EmployeeId
    ) AS d
    PIVOT (COUNT(OrderYear) FOR OrderYear IN ([2020],[2021],[2022],[2023])) AS p
) AS pivoted
UNPIVOT (NumOrders FOR OrderYear IN ([2020],[2021],[2022],[2023])) AS unpvt
WHERE NumOrders > 0;


Proposition 7 — Grouping Sets for Multiple Totals

Outer Query:

- Calculates total sales grouped by employee, country, and year.

Logic:

- GROUPING SETS produces multiple levels of aggregation in one query.

Result:

- Gives per employee, per country, and per year subtotals in a single dataset.

In [ ]:
%%sql
SELECT
    e.EmployeeId,
    c.CustomerCountry,
    YEAR(o.OrderDate) AS OrderYear,
    SUM(od.UnitPrice * od.Quantity) AS TotalSales
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
JOIN HumanResources.Employee AS e ON o.EmployeeId = e.EmployeeId
JOIN Sales.Customer AS c ON o.CustomerId = c.CustomerId
GROUP BY GROUPING SETS (
    (e.EmployeeId, c.CustomerCountry, YEAR(o.OrderDate)),
    (e.EmployeeId, YEAR(o.OrderDate)),
    (c.CustomerCountry, YEAR(o.OrderDate))
);


Proposition 8 — ROLLUP by Year, Quarter, and Month

Outer Query:

- Aggregates sales by year, quarter, and month.

Logic:

- ROLLUP adds subtotal levels automatically for hierarchical time analysis.

Result:

- Shows detailed and subtotaled sales progression for each period.

In [ ]:
%%sql
SELECT
    YEAR(o.OrderDate) AS OrderYear,
    DATEPART(QUARTER, o.OrderDate) AS OrderQuarter,
    MONTH(o.OrderDate) AS OrderMonth,
    SUM(od.UnitPrice * od.Quantity) AS TotalSales
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
GROUP BY ROLLUP (YEAR(o.OrderDate), DATEPART(QUARTER, o.OrderDate), MONTH(o.OrderDate))
ORDER BY OrderYear, OrderQuarter, OrderMonth;


Proposition 9 — Pivot Customer Totals by Employee

Outer Query:

- Summarizes quantities sold by employee and customer.

Logic:

- Rotates customer countries into columns with PIVOT.

Result:

- Creates a quick view of employee performance across markets.

In [ ]:
%%sql
SELECT *
FROM (
    SELECT e.EmployeeLastName, c.CustomerCountry, od.Quantity
    FROM Sales.[Order] AS o
    JOIN HumanResources.Employee AS e ON o.EmployeeId = e.EmployeeId
    JOIN Sales.Customer AS c ON o.CustomerId = c.CustomerId
    JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
) AS src
PIVOT (
    SUM(Quantity) FOR CustomerCountry IN ([USA], [UK], [France], [Germany])
) AS p;


Proposition 10 — Using CUBE for All Subtotals

Outer Query:

- Computes all possible combinations of employee, country, and year subtotals.

Logic:

- CUBE produces grand totals and partial totals.
- GROUPING_ID identifies which grouping each row belongs to.

Result:

- Displays total quantity sold at every aggregation level.

In [ ]:
%%sql
SELECT
    e.EmployeeId,
    c.CustomerCountry,
    YEAR(o.OrderDate) AS OrderYear,
    SUM(od.Quantity) AS TotalQuantity,
    GROUPING_ID(e.EmployeeId, c.CustomerCountry, YEAR(o.OrderDate)) AS GroupingSetId
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
JOIN Sales.Customer AS c ON o.CustomerId = c.CustomerId
JOIN HumanResources.Employee AS e ON o.EmployeeId = e.EmployeeId
GROUP BY CUBE (e.EmployeeId, c.CustomerCountry, YEAR(o.OrderDate))
ORDER BY GroupingSetId, e.EmployeeId;
